# Aether Stage 3 training (remote GPU)

Clones the repo and runs the existing scripts from `scripts/` cell by cell. No logic is duplicated here — each cell just shells out to a script that already lives in the repo. Run cells top to bottom.

In [ ]:
!git clone https://github.com/karl4th/aether.git
%cd aether

## Environment setup (uv)

In [ ]:
!pip install -q uv
!uv venv .venv
!uv pip install -r requirements.txt

Check CUDA. If `torch.cuda.is_available()` is `False`, run `nvidia-smi` to see the driver's CUDA version, then reinstall torch from the matching PyTorch index in the next cell (cu121/cu124/cu126/cu128/cu129) before continuing.

In [ ]:
!nvidia-smi
!uv run python -c "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())"

In [ ]:
# Only run this if the cell above showed CUDA not available.
# Swap cu126 for whatever index matches your machine's driver.
!uv pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu126 --reinstall-package torch --reinstall-package torchaudio

## Stage 2: build the dataset (20,000 sentences)

Each script skips files that already exist by id, so re-running any cell after an interruption is safe and resumes instead of restarting.

In [ ]:
!uv run python -m piper.download_voices en_US-lessac-medium --download-dir data/piper_voices

In [ ]:
!uv run python scripts/build_sentences.py --num-sentences 20000

In [ ]:
!uv run python scripts/synthesize_tts.py

In [ ]:
!uv run python scripts/extract_hidden_states.py

## Stage 3: train the decoder

`--val-size 1000` keeps roughly the same ~5% held-out fraction as the earlier 1500-sentence run. Checkpoints (`best.pt`, periodic `epoch_NNNN.pt`, `last.pt`) are written to `checkpoints/decoder/` as training goes, so state survives a disconnect if the runtime's filesystem persists (or if you mount Drive / a volume there).

In [ ]:
!uv run python scripts/train_decoder.py --epochs 50 --val-size 1000

## Listen to a result

In [ ]:
!uv run python scripts/listen_to_decoder.py --id 00042

In [ ]:
from IPython.display import Audio, display

print("Predicted:")
display(Audio("data/samples/00042.wav"))
print("Ground truth (Piper):")
display(Audio("data/dataset/audio/00042.wav"))

## Download the trained checkpoint

Zips `checkpoints/decoder/` so you can pull it back to your own machine (via the file browser's download, or `files.download(...)` on Colab).

In [ ]:
!zip -r decoder_checkpoints.zip checkpoints/decoder